# Comparative Evaluation: Three Methods on Crop Disease UAV Monitoring

Runs three methods **sequentially** in one notebook and compares them.

| # | Method | Algorithm | Environment | Networks |
|---|--------|-----------|-------------|----------|
| 1 | **Our MAPPO** (main paper) | Multi-Agent PPO, per-UAV critic heads, GAE | `uav_env_5` | `networks_5` — SectorAttentionActor + CriticNetwork |
| 2 | **Pareto Multi-Mode MAPPO** | 3 specialized PPO policies + rule-based mode selector (LLM analogue) | `uav_env_5` | `networks_5` — same architecture, 3 actor instances |
| 3 | **M2LLM Centralized DDPG** | Single centralized DDPG, LLMEncoder state projection | `uav_env_beamforming` | `networks_beamforming` — LLMEncoder + CentralDDPGActor/Critic |

**Reward fixes applied to both environments** (vs old fixes_10 results):
- `INFECTED_FOUND_BONUS` 15 → 30: detection now outweighs crash risk
- `END_OF_DAY_CRASH_PENALTY` = 5 (new): time-limit violations cost 5, not 20
- `CRASH_PENALTY` = 20 kept for real energy crashes
- `STEP_ALIVE_REWARD` 0.05 → 0.01: reduced stay-home incentive
- `SAFE_RETURN_BONUS` 3 → 5

Set `QUICK_MODE = True` to run 50 episodes per method for a fast sanity check.

In [ ]:
import os, sys, time, random, math, collections
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.distributions import Normal
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Path setup ───────────────────────────────────────────────────────────────
_CWD      = os.getcwd()
PROJ      = _CWD if os.path.isdir(os.path.join(_CWD, 'fixes-2')) else os.path.dirname(os.path.abspath('comparison.ipynb'))
FIXES2    = os.path.join(PROJ, 'fixes-2')
NEW_IMPL  = os.path.join(PROJ, 'new-implementation')
if not os.path.isdir(FIXES2) or not os.path.isdir(NEW_IMPL):
    raise FileNotFoundError('Run this notebook from the project directory containing fixes-2/ and new-implementation/.')
for p in [FIXES2, NEW_IMPL]:
    if p not in sys.path:
        sys.path.insert(0, p)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('fixes-2  :', FIXES2)
print('new-impl :', NEW_IMPL)

In [ ]:
# ── Global settings ──────────────────────────────────────────────────────────
QUICK_MODE = False          # True → 50 eps per method (fast sanity check)
EPISODES   = 50 if QUICK_MODE else 400
SEED       = 42
print(f'Training for {EPISODES} episodes per method  [QUICK_MODE={QUICK_MODE}]')

---
## Method 1 — Our MAPPO
**Multi-Agent PPO** with per-UAV critic heads and attention-based actor.
- Environment: `uav_env_5` (identical to main paper)
- Actor: `SectorAttentionActor` (2-layer Transformer over 100 sector tokens)
- Critic: `CriticNetwork` (shared trunk + 4 per-UAV value heads)
- Training: per-episode std normalization, KL early-stop, entropy annealing

In [ ]:
from uav_env_5 import (
    UAVFieldEnv, N_UAVS, N_SECTORS, OBS_SIZE, JOINT_SIZE, ENV_OBS_DIM,
    GRID_ROWS, GRID_COLS,
)
from networks_5 import SectorAttentionActor, CriticNetwork
print('Method 1 imports OK')
print(f'  OBS_SIZE={OBS_SIZE}  JOINT_SIZE={JOINT_SIZE}')

In [ ]:
# Hyperparameters — Method 1
M1_GAMMA      = 0.99
M1_GAE_LAM    = 0.95
M1_K_EPOCHS   = 2
M1_MINI_BATCH = 128
M1_CLIP_EPS   = 0.2
M1_MAX_GRAD   = 0.5
M1_REWARD_CLIP = 50.0
M1_KL_THRESH  = 0.025
M1_ENT_START  = 0.02    # higher than before to encourage early exploration
M1_ENT_END    = 0.001

def m1_ent_coeff(ep):
    t = ep / max(EPISODES - 1, 1)
    return M1_ENT_START * (1 - t) + M1_ENT_END * t


class RolloutBuffer:
    def __init__(self):
        self.obs       = [[] for _ in range(N_UAVS)]
        self.actions   = [[] for _ in range(N_UAVS)]
        self.log_probs = [[] for _ in range(N_UAVS)]
        self.rewards   = [[] for _ in range(N_UAVS)]
        self.dones     = []
        self.alive     = [[] for _ in range(N_UAVS)]
    def clear(self): self.__init__()


def compute_gae_m1(rewards_u, vals_u, dones, alive_u, gamma, lam):
    T = len(rewards_u)
    adv = torch.zeros(T, device=DEVICE)
    gae = 0.0
    vals = vals_u.detach()
    for t in reversed(range(T)):
        mask  = 1.0 - float(dones[t])
        nv    = vals[t + 1] if t + 1 < len(vals) else 0.0
        delta = rewards_u[t] + gamma * nv * mask - vals[t]
        gae   = delta + gamma * lam * mask * gae
        adv[t] = gae
    ret = adv + vals[:T]
    alive_t = torch.tensor(alive_u, dtype=torch.float32, device=DEVICE)
    if alive_t.sum() > 1:
        adv = (adv - adv[alive_t.bool()].mean()) / (adv[alive_t.bool()].std() + 1e-8)
    return adv, ret


def m1_ppo_update(actor, critic, a_opt, c_opt, buf, ep):
    ent_c = m1_ent_coeff(ep)
    T = len(buf.dones)
    all_rews = np.concatenate([buf.rewards[u] for u in range(N_UAVS)])
    scale    = max(float(np.std(all_rews)), 1.0)

    joint_obs_t = torch.tensor(
        np.stack([np.concatenate([buf.obs[u][t] for u in range(N_UAVS)])
                  for t in range(T)]),
        dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        all_vals = critic(joint_obs_t)  # (T, N_UAVS)

    adv_list, ret_list = [], []
    for u in range(N_UAVS):
        rn = [float(np.clip(r, -M1_REWARD_CLIP, M1_REWARD_CLIP)) / scale
              for r in buf.rewards[u]]
        adv_u, ret_u = compute_gae_m1(rn, all_vals[:, u], buf.dones,
                                       buf.alive[u], M1_GAMMA, M1_GAE_LAM)
        adv_list.append(adv_u)
        ret_list.append(ret_u)

    obs_t    = [torch.tensor(np.array(buf.obs[u]),     dtype=torch.float32, device=DEVICE) for u in range(N_UAVS)]
    act_t    = [torch.tensor(np.array(buf.actions[u]), dtype=torch.float32, device=DEVICE) for u in range(N_UAVS)]
    lp_old_t = [torch.tensor(np.array(buf.log_probs[u]), dtype=torch.float32, device=DEVICE) for u in range(N_UAVS)]
    alive_t  = [torch.tensor(buf.alive[u], dtype=torch.float32, device=DEVICE) for u in range(N_UAVS)]

    idx = np.arange(T)
    for _ in range(M1_K_EPOCHS):
        np.random.shuffle(idx)
        for start in range(0, T, M1_MINI_BATCH):
            mb   = idx[start:start + M1_MINI_BATCH]
            mb_t = torch.tensor(mb, device=DEVICE)
            a_loss = torch.tensor(0.0, device=DEVICE)
            kl_sum = 0.0
            for u in range(N_UAVS):
                lp_new, ent = actor.get_log_prob_entropy(obs_t[u][mb_t], act_t[u][mb_t])
                ratio  = torch.exp(lp_new - lp_old_t[u][mb_t])
                surr1  = ratio * adv_list[u][mb_t]
                surr2  = torch.clamp(ratio, 1 - M1_CLIP_EPS, 1 + M1_CLIP_EPS) * adv_list[u][mb_t]
                mask_u = alive_t[u][mb_t]
                a_loss += (-torch.min(surr1, surr2) * mask_u).sum() / mask_u.sum().clamp(min=1)
                a_loss -= ent_c * ent
                with torch.no_grad():
                    kl_sum += (lp_old_t[u][mb_t] - lp_new).mean().item()
            if kl_sum / N_UAVS > M1_KL_THRESH:
                break
            a_opt.zero_grad()
            (a_loss / N_UAVS).backward()
            nn.utils.clip_grad_norm_(actor.parameters(), M1_MAX_GRAD)
            a_opt.step()
            c_loss = sum(
                nn.functional.mse_loss(critic(joint_obs_t[mb_t])[:, u], ret_list[u][mb_t])
                for u in range(N_UAVS)) / N_UAVS
            c_opt.zero_grad()
            c_loss.backward()
            nn.utils.clip_grad_norm_(critic.parameters(), M1_MAX_GRAD)
            c_opt.step()

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

m1_actor  = SectorAttentionActor().to(DEVICE)
m1_critic = CriticNetwork().to(DEVICE)
m1_a_opt  = Adam(m1_actor.parameters(),  lr=3e-4)
m1_c_opt  = Adam(m1_critic.parameters(), lr=1e-3)

m1_env = UAVFieldEnv(seed=SEED)

m1_rewards, m1_infected, m1_diagnosed, m1_crashes, m1_det_rate = [], [], [], [], []

t0 = time.time()
print('=== Method 1: Our MAPPO ===')
for ep in range(EPISODES):
    obs_list = m1_env.reset()
    buf = RolloutBuffer()
    ep_r = 0.0
    crashes_ep = 0

    for _ in range(m1_env.total_steps):
        obs_t = [torch.tensor(obs_list[u], dtype=torch.float32, device=DEVICE) for u in range(N_UAVS)]
        actions_np, log_probs = [], []
        for u in range(N_UAVS):
            if m1_env.crashed[u]:
                actions_np.append(np.zeros(2, dtype=np.float32))
                log_probs.append(torch.tensor(0.0, device=DEVICE))
            else:
                act, lp = m1_actor.get_action(obs_t[u])
                actions_np.append(act)
                log_probs.append(lp.squeeze())

        next_obs, rewards, done, info = m1_env.step(actions_np)
        for u in range(N_UAVS):
            buf.obs[u].append(obs_list[u])
            buf.actions[u].append(actions_np[u])
            buf.log_probs[u].append(log_probs[u].item())
            buf.rewards[u].append(rewards[u])
            buf.alive[u].append(0.0 if m1_env.crashed[u] else 1.0)
        buf.dones.append(done)

        ep_r += sum(rewards)
        crashes_ep += sum(info['newly_crashed'])
        obs_list = next_obs
        if done: break

    m1_ppo_update(m1_actor, m1_critic, m1_a_opt, m1_c_opt, buf, ep)

    ever_inf  = int(m1_env.ever_infected.sum())
    diag_inf  = int((m1_env.ever_diagnosed & m1_env.ever_infected).sum())
    det_rate  = diag_inf / max(ever_inf, 1)
    m1_rewards.append(ep_r)
    m1_infected.append(float(m1_env.true_status.sum()) / N_SECTORS)
    m1_diagnosed.append(float(m1_env.ever_diagnosed.sum()) / N_SECTORS)
    m1_crashes.append(crashes_ep)
    m1_det_rate.append(det_rate)

    if (ep + 1) % 50 == 0:
        print(f'  Ep {ep+1:4d}/{EPISODES}  rew={ep_r:8.1f}  '
              f'det={det_rate:.1%}  diag={m1_env.ever_diagnosed.sum()}/100  '
              f'inf={m1_infected[-1]:.2f}  crash={crashes_ep}  '
              f'{time.time()-t0:.0f}s')

print(f'\nMethod 1 done. Final-50 det={np.mean(m1_det_rate[-50:]):.1%}  '
      f'crash={np.mean(m1_crashes[-50:]):.1f}  '
      f'diag={np.mean(m1_diagnosed[-50:]):.3f}  time={time.time()-t0:.0f}s')

---
## Method 2 — Pareto Multi-Mode MAPPO
Adapted from: *LLM-Driven Pareto-Optimal Multi-Mode RL*.
- Environment: `uav_env_5` (same as Method 1 — fair comparison)
- 3 specialized actors: Aggressive, Balanced, Cautious
- Hard-coded telemetry threshold selector at episode start (simulates LLM meta-selector)
- Mode-specific reward scaling applied to raw env rewards
- Shared centralized critic (per-UAV heads, same `CriticNetwork`)

In [ ]:
# Networks already imported from networks_5 (same file as Method 1)
MODES = {
    'aggressive': {'detection': 3.0, 'task': 2.0, 'safety': 0.5},
    'balanced':   {'detection': 1.5, 'task': 1.0, 'safety': 1.0},
    'cautious':   {'detection': 0.5, 'task': 0.5, 'safety': 2.0},
}
MODE_NAMES = list(MODES.keys())


def build_selector_context(env):
    """Telemetry context used by the hard-coded LLM selector analogue."""
    infected_frac = float(env.true_status.sum()) / N_SECTORS
    avg_energy = float(np.mean(env.energy)) / 150.0
    weather_risk = max(
        float(env.wind_speed) / 12.0,
        float(env.humidity) / 100.0,
        float(env.season_mult) / 1.5,
    )
    return {
        'infected_frac': infected_frac,
        'avg_energy': avg_energy,
        'wind_speed': float(env.wind_speed),
        'humidity': float(env.humidity),
        'season_mult': float(env.season_mult),
        'weather_risk': weather_risk,
    }


def select_mode(context: dict) -> str:
    """Hard-coded threshold selector used in place of a fine-tuned LLM."""
    if context['avg_energy'] < 0.35 or context['wind_speed'] >= 8.5:
        return 'cautious'
    if (context['infected_frac'] >= 0.03 or
            context['humidity'] >= 75.0 or
            context['season_mult'] >= 1.15):
        return 'aggressive'
    if context['weather_risk'] >= 0.55:
        return 'balanced'
    return 'cautious'


def scale_reward_pareto(raw_reward, comp, mp):
    r  = raw_reward
    r -= comp.get('discovery_bonus', 0.0)
    r += mp['detection'] * comp.get('discovery_bonus', 0.0)
    r += comp.get('crash_penalty', 0.0)
    r -= mp['safety'] * comp.get('crash_penalty', 0.0)
    r -= comp.get('safe_return_bonus', 0.0)
    r += mp['task'] * comp.get('safe_return_bonus', 0.0)
    return r


# Hyperparameters — Method 2 (same PPO settings as Method 1)
M2_KL_THRESH  = 0.025
M2_ENT_START  = 0.02
M2_ENT_END    = 0.001

def m2_ent_coeff(ep):
    t = ep / max(EPISODES - 1, 1)
    return M2_ENT_START * (1 - t) + M2_ENT_END * t


def m2_ppo_update(mode, actors, critic, actor_opts, c_opt, buf, ep):
    actor  = actors[mode]
    a_opt  = actor_opts[mode]
    ent_c  = m2_ent_coeff(ep)
    T      = len(buf.dones)
    all_rews = np.concatenate([buf.rewards[u] for u in range(N_UAVS)])
    scale    = max(float(np.std(all_rews)), 1.0)

    joint_obs_t = torch.tensor(
        np.stack([np.concatenate([buf.obs[u][t] for u in range(N_UAVS)])
                  for t in range(T)]),
        dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        all_vals = critic(joint_obs_t)

    adv_list, ret_list = [], []
    for u in range(N_UAVS):
        rn = [float(np.clip(r, -M1_REWARD_CLIP, M1_REWARD_CLIP)) / scale
              for r in buf.rewards[u]]
        adv_u, ret_u = compute_gae_m1(rn, all_vals[:, u], buf.dones,
                                       buf.alive[u], M1_GAMMA, M1_GAE_LAM)
        adv_list.append(adv_u)
        ret_list.append(ret_u)

    obs_t    = [torch.tensor(np.array(buf.obs[u]),     dtype=torch.float32, device=DEVICE) for u in range(N_UAVS)]
    act_t    = [torch.tensor(np.array(buf.actions[u]), dtype=torch.float32, device=DEVICE) for u in range(N_UAVS)]
    lp_old_t = [torch.tensor(np.array(buf.log_probs[u]), dtype=torch.float32, device=DEVICE) for u in range(N_UAVS)]
    alive_t  = [torch.tensor(buf.alive[u], dtype=torch.float32, device=DEVICE) for u in range(N_UAVS)]

    idx = np.arange(T)
    for _ in range(M1_K_EPOCHS):
        np.random.shuffle(idx)
        for start in range(0, T, M1_MINI_BATCH):
            mb   = idx[start:start + M1_MINI_BATCH]
            mb_t = torch.tensor(mb, device=DEVICE)
            a_loss = torch.tensor(0.0, device=DEVICE)
            kl_sum = 0.0
            for u in range(N_UAVS):
                lp_new, ent = actor.get_log_prob_entropy(obs_t[u][mb_t], act_t[u][mb_t])
                ratio  = torch.exp(lp_new - lp_old_t[u][mb_t])
                surr1  = ratio * adv_list[u][mb_t]
                surr2  = torch.clamp(ratio, 1 - M1_CLIP_EPS, 1 + M1_CLIP_EPS) * adv_list[u][mb_t]
                mask_u = alive_t[u][mb_t]
                a_loss += (-torch.min(surr1, surr2) * mask_u).sum() / mask_u.sum().clamp(min=1)
                a_loss -= ent_c * ent
                with torch.no_grad():
                    kl_sum += (lp_old_t[u][mb_t] - lp_new).mean().item()
            if kl_sum / N_UAVS > M2_KL_THRESH:
                break
            a_opt.zero_grad()
            (a_loss / N_UAVS).backward()
            nn.utils.clip_grad_norm_(actor.parameters(), M1_MAX_GRAD)
            a_opt.step()
            c_loss = sum(
                nn.functional.mse_loss(critic(joint_obs_t[mb_t])[:, u], ret_list[u][mb_t])
                for u in range(N_UAVS)) / N_UAVS
            c_opt.zero_grad()
            c_loss.backward()
            nn.utils.clip_grad_norm_(critic.parameters(), M1_MAX_GRAD)
            c_opt.step()

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

m2_actors     = {m: SectorAttentionActor().to(DEVICE) for m in MODE_NAMES}
m2_critic     = CriticNetwork().to(DEVICE)
m2_actor_opts = {m: Adam(m2_actors[m].parameters(), lr=3e-4) for m in MODE_NAMES}
m2_c_opt      = Adam(m2_critic.parameters(), lr=1e-3)

m2_env = UAVFieldEnv(seed=SEED)

m2_rewards, m2_infected, m2_diagnosed, m2_crashes, m2_det_rate = [], [], [], [], []
m2_mode_counts = {m: 0 for m in MODE_NAMES}

t0 = time.time()
print('=== Method 2: Pareto Multi-Mode MAPPO ===')
for ep in range(EPISODES):
    obs_list = m2_env.reset()

    selector_ctx  = build_selector_context(m2_env)
    mode          = select_mode(selector_ctx)
    mp            = MODES[mode]
    actor         = m2_actors[mode]
    m2_mode_counts[mode] += 1

    buf = RolloutBuffer()
    ep_r = 0.0
    crashes_ep = 0

    for _ in range(m2_env.total_steps):
        obs_t = [torch.tensor(obs_list[u], dtype=torch.float32, device=DEVICE) for u in range(N_UAVS)]
        actions_np, log_probs = [], []
        for u in range(N_UAVS):
            if m2_env.crashed[u]:
                actions_np.append(np.zeros(2, dtype=np.float32))
                log_probs.append(torch.tensor(0.0, device=DEVICE))
            else:
                act, lp = actor.get_action(obs_t[u])
                actions_np.append(act)
                log_probs.append(lp.squeeze())

        next_obs, raw_rews, done, info = m2_env.step(actions_np)
        comps   = info['reward_components']
        rewards = [scale_reward_pareto(raw_rews[u], comps[u], mp) for u in range(N_UAVS)]

        for u in range(N_UAVS):
            buf.obs[u].append(obs_list[u])
            buf.actions[u].append(actions_np[u])
            buf.log_probs[u].append(log_probs[u].item())
            buf.rewards[u].append(rewards[u])
            buf.alive[u].append(0.0 if m2_env.crashed[u] else 1.0)
        buf.dones.append(done)

        ep_r += sum(raw_rews)
        crashes_ep += sum(info['newly_crashed'])
        obs_list = next_obs
        if done: break

    m2_ppo_update(mode, m2_actors, m2_critic, m2_actor_opts, m2_c_opt, buf, ep)

    ever_inf = int(m2_env.ever_infected.sum())
    diag_inf = int((m2_env.ever_diagnosed & m2_env.ever_infected).sum())
    m2_rewards.append(ep_r)
    m2_infected.append(float(m2_env.true_status.sum()) / N_SECTORS)
    m2_diagnosed.append(float(m2_env.ever_diagnosed.sum()) / N_SECTORS)
    m2_crashes.append(crashes_ep)
    m2_det_rate.append(diag_inf / max(ever_inf, 1))

    if (ep + 1) % 50 == 0:
        print(f'  Ep {ep+1:4d}/{EPISODES}  mode={mode:10s}  rew={ep_r:8.1f}  '
              f'det={m2_det_rate[-1]:.1%}  crash={crashes_ep}  {time.time()-t0:.0f}s')

print(f'\nMethod 2 done. Final-50 det={np.mean(m2_det_rate[-50:]):.1%}  '
      f'crash={np.mean(m2_crashes[-50:]):.1f}  modes={m2_mode_counts}  time={time.time()-t0:.0f}s')

---
## Method 3 — M2LLM Centralized DDPG
Adapted from: *Trajectory Design and Beamforming with M2LLM-Driven DRL*.
- Environment: `uav_env_beamforming` (extended env with treatment intensity action)
- Architecture: `LLMEncoder` (LayerNorm + GELU deep MLP) → centralized DDPG
- Single agent controlling all 4 UAVs jointly (12-dim action: [vx,vy,intensity]×4)
- Replay buffer, target networks, OUNoise, soft τ-update
- Note: uses `uav_env_beamforming` (same disease dynamics, added intensity action)

In [ ]:
from uav_env_beamforming import (
    UAVFieldEnvBeamforming,
    N_UAVS as BF_N_UAVS,
    N_SECTORS as BF_N_SECTORS,
    OBS_SIZE as BF_OBS_SIZE,
    JOINT_SIZE as BF_JOINT_SIZE,
    ACTION_DIM, JOINT_ACTION_DIM,
)
from networks_beamforming import LLMEncoder, CentralDDPGActor, CentralDDPGCritic
print('Method 3 imports OK')
print(f'  BF_JOINT_SIZE={BF_JOINT_SIZE}  JOINT_ACTION_DIM={JOINT_ACTION_DIM}')
assert BF_N_UAVS == N_UAVS and BF_N_SECTORS == N_SECTORS, 'env mismatch'
print('  env constants match uav_env_5: OK')

In [ ]:
# Hyperparameters — Method 3
M3_GAMMA       = 0.99
M3_TAU         = 0.005
M3_BATCH       = 256
M3_WARMUP      = 2000
M3_REWARD_CLIP = 50.0
M3_NOISE_DECAY = 0.999
M3_NOISE_MIN   = 0.05


class ReplayBuffer:
    def __init__(self, cap=200_000):
        self.buf = collections.deque(maxlen=cap)
    def push(self, s, a, r, s2, d):
        self.buf.append((s, a, r, s2, d))
    def sample(self, bs):
        batch = random.sample(self.buf, bs)
        s, a, r, s2, d = map(np.array, zip(*batch))
        return (torch.tensor(s,  dtype=torch.float32, device=DEVICE),
                torch.tensor(a,  dtype=torch.float32, device=DEVICE),
                torch.tensor(r,  dtype=torch.float32, device=DEVICE).unsqueeze(1),
                torch.tensor(s2, dtype=torch.float32, device=DEVICE),
                torch.tensor(d,  dtype=torch.float32, device=DEVICE).unsqueeze(1))
    def __len__(self): return len(self.buf)


class OUNoise:
    def __init__(self, dim, mu=0.0, theta=0.15, sigma=0.2):
        self.dim = dim; self.mu = mu; self.theta = theta
        self.sigma = sigma; self.state = np.zeros(dim)
    def reset(self): self.state = np.zeros(self.dim)
    def sample(self):
        dx = self.theta * (self.mu - self.state) + self.sigma * np.random.randn(self.dim)
        self.state += dx
        return self.state.copy()


def ddpg_update_m3(actor, critic, t_actor, t_critic, a_opt, c_opt, replay):
    if len(replay) < M3_BATCH:
        return None, None
    s, a, r, s2, done = replay.sample(M3_BATCH)
    with torch.no_grad():
        q_next = t_critic(s2, t_actor(s2))
        y = torch.clamp(r, -M3_REWARD_CLIP, M3_REWARD_CLIP) + M3_GAMMA * q_next * (1 - done)
    c_loss = F.mse_loss(critic(s, a), y)
    c_opt.zero_grad(); c_loss.backward()
    nn.utils.clip_grad_norm_(critic.parameters(), 1.0)
    c_opt.step()
    a_loss = -critic(s, actor(s)).mean()
    a_opt.zero_grad(); a_loss.backward()
    nn.utils.clip_grad_norm_(actor.parameters(), 1.0)
    a_opt.step()
    for tp, p in zip(t_actor.parameters(),  actor.parameters()):
        tp.data.mul_(1 - M3_TAU).add_(M3_TAU * p.data)
    for tp, p in zip(t_critic.parameters(), critic.parameters()):
        tp.data.mul_(1 - M3_TAU).add_(M3_TAU * p.data)
    return c_loss.item(), a_loss.item()

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

m3_actor         = CentralDDPGActor().to(DEVICE)
m3_critic        = CentralDDPGCritic().to(DEVICE)
m3_target_actor  = CentralDDPGActor().to(DEVICE)
m3_target_critic = CentralDDPGCritic().to(DEVICE)
m3_target_actor.load_state_dict(m3_actor.state_dict())
m3_target_critic.load_state_dict(m3_critic.state_dict())
m3_a_opt = Adam(m3_actor.parameters(),  lr=1e-4)
m3_c_opt = Adam(m3_critic.parameters(), lr=1e-3)
m3_replay = ReplayBuffer()
m3_noise  = OUNoise(JOINT_ACTION_DIM)

m3_env = UAVFieldEnvBeamforming(seed=SEED)

m3_rewards, m3_infected, m3_diagnosed, m3_crashes, m3_det_rate = [], [], [], [], []
global_step_m3 = 0

t0 = time.time()
print('=== Method 3: M2LLM Centralized DDPG ===')
for ep in range(EPISODES):
    obs_list  = m3_env.reset()
    joint_obs = np.concatenate(obs_list).astype(np.float32)
    m3_noise.reset()
    m3_noise.sigma = max(m3_noise.sigma * M3_NOISE_DECAY, M3_NOISE_MIN)

    ep_r = 0.0
    crashes_ep = 0

    for _ in range(m3_env.total_steps):
        if global_step_m3 < M3_WARMUP:
            jact = np.random.uniform(-1, 1, JOINT_ACTION_DIM).astype(np.float32)
        else:
            with torch.no_grad():
                ot = torch.tensor(joint_obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                jact = m3_actor(ot).squeeze(0).cpu().numpy()
            jact = np.clip(jact + m3_noise.sample(), -1.0, 1.0)

        next_obs_list, rewards, done, info = m3_env.step(jact)
        next_joint = np.concatenate(next_obs_list).astype(np.float32)
        total_r    = float(sum(rewards))
        m3_replay.push(joint_obs, jact, total_r, next_joint, float(done))
        global_step_m3 += 1
        ddpg_update_m3(m3_actor, m3_critic, m3_target_actor, m3_target_critic,
                       m3_a_opt, m3_c_opt, m3_replay)
        ep_r += total_r
        crashes_ep += sum(info['newly_crashed'])
        joint_obs = next_joint
        if done: break

    ever_inf = int(m3_env.ever_infected.sum())
    diag_inf = int((m3_env.ever_diagnosed & m3_env.ever_infected).sum())
    m3_rewards.append(ep_r)
    m3_infected.append(float(m3_env.true_status.sum()) / N_SECTORS)
    m3_diagnosed.append(float(m3_env.ever_diagnosed.sum()) / N_SECTORS)
    m3_crashes.append(crashes_ep)
    m3_det_rate.append(diag_inf / max(ever_inf, 1))

    if (ep + 1) % 50 == 0:
        print(f'  Ep {ep+1:4d}/{EPISODES}  rew={ep_r:8.1f}  '
              f'det={m3_det_rate[-1]:.1%}  crash={crashes_ep}  {time.time()-t0:.0f}s')

print(f'\nMethod 3 done. Final-50 det={np.mean(m3_det_rate[-50:]):.1%}  '
      f'crash={np.mean(m3_crashes[-50:]):.1f}  time={time.time()-t0:.0f}s')

---
## Comparative Analysis
Comparing all three methods across the key metrics relevant to crop disease monitoring.

In [ ]:
# ── Smoothing helper ─────────────────────────────────────────────────────────
def smooth(x, w=20):
    x = np.array(x, dtype=float)
    if len(x) < w: return x
    return np.convolve(x, np.ones(w) / w, mode='valid')

LABELS  = ['Our MAPPO', 'Pareto MAPPO', 'M2LLM DDPG']
COLORS  = ['tab:blue', 'tab:orange', 'tab:green']
LINEST  = ['-', '--', '-.']

all_rewards   = [m1_rewards,   m2_rewards,   m3_rewards]
all_det_rate  = [m1_det_rate,  m2_det_rate,  m3_det_rate]
all_diagnosed = [m1_diagnosed, m2_diagnosed, m3_diagnosed]
all_crashes   = [m1_crashes,   m2_crashes,   m3_crashes]
all_infected  = [m1_infected,  m2_infected,  m3_infected]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Comparative Training Curves — 3 Methods', fontsize=15, fontweight='bold')

PLOT_DEF = [
    (axes[0,0], all_rewards,   'Episode Reward (sum all UAVs)',       None),
    (axes[0,1], all_det_rate,  'Detection Rate (infected sectors %)', None),
    (axes[0,2], all_diagnosed, 'Diagnosed Fraction',                  None),
    (axes[1,0], all_crashes,   'Crash Events per Episode',            None),
    (axes[1,1], all_infected,  'Disease Fraction at End of Episode',  0.35),
]

for ax, data_list, title, hline in PLOT_DEF:
    for i, (data, lbl, col, ls) in enumerate(zip(data_list, LABELS, COLORS, LINEST)):
        s = smooth(data)
        x = np.arange(len(s)) + 20  # offset by smoothing window
        ax.plot(x, s, color=col, linestyle=ls, label=lbl)
    if hline:
        ax.axhline(hline, color='gray', linestyle=':', linewidth=1, label='endemic baseline')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Episode')
    ax.legend(fontsize=8)

# Unused 6th panel — mode breakdown for Method 2
ax6 = axes[1,2]
ax6.bar(list(m2_mode_counts.keys()), list(m2_mode_counts.values()),
        color=['tab:red', 'tab:orange', 'tab:cyan'])
ax6.set_title('Method 2: Mode Selection Count')
ax6.set_ylabel('Episodes using mode')

plt.tight_layout()
plt.savefig('comparison_training.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved comparison_training.png')

In [ ]:
# ── Summary statistics table ─────────────────────────────────────────────────
def stats(arr, n=50):
    a = np.array(arr[-n:])
    return float(np.mean(a)), float(np.std(a)), float(np.max(a))

print('\n' + '='*72)
print(f'  COMPARATIVE RESULTS — Final {min(50, EPISODES)} Episodes')
print('='*72)

metrics = [
    ('Detection Rate (%)',    all_det_rate,   True,  100.0),
    ('Diagnosed Fraction',    all_diagnosed,  True,  1.0),
    ('Crashes / Episode',     all_crashes,    False, 1.0),
    ('Disease Frac at End',   all_infected,   False, 1.0),
    ('Episode Reward',        all_rewards,    True,  1.0),
]

print(f'  {"Metric":<28}  {"Our MAPPO":>14}  {"Pareto MAPPO":>14}  {"M2LLM DDPG":>14}  Best')
print('-'*72)

for metric_name, data_list, higher_better, scale in metrics:
    vals = [stats(d)[0] * scale for d in data_list]
    best_idx = (np.argmax if higher_better else np.argmin)(vals)
    stars = ['★' if i == best_idx else ' ' for i in range(3)]
    print(f'  {metric_name:<28}  '
          f'{vals[0]:>13.2f}{stars[0]}  '
          f'{vals[1]:>13.2f}{stars[1]}  '
          f'{vals[2]:>13.2f}{stars[2]}  {LABELS[best_idx]}')

print('='*72)
print('★ = best performer for that metric')

In [ ]:
# ── Per-method final summary ─────────────────────────────────────────────────
print('\nDetailed per-method final-50 summary:')
for name, rew, det, diag, crash, inf in zip(
        LABELS, all_rewards, all_det_rate, all_diagnosed, all_crashes, all_infected):
    r_m, r_s, _ = stats(rew)
    d_m, d_s, _ = stats(det)
    g_m, _, _   = stats(diag)
    c_m, _, _   = stats(crash)
    i_m, _, _   = stats(inf)
    print(f'\n  [{name}]')
    print(f'    Reward       : {r_m:8.1f} ± {r_s:.1f}')
    print(f'    Detection    : {d_m:.1%} ± {d_s:.1%}')
    print(f'    Diagnosed    : {g_m:.3f}  (fraction of 100 sectors)')
    print(f'    Crashes/ep   : {c_m:.1f}')
    print(f'    Disease frac : {i_m:.3f}')

In [ ]:
# ── Save weights ─────────────────────────────────────────────────────────────
SAVE_DIR = '.'
torch.save(m1_actor.state_dict(),  os.path.join(SAVE_DIR, 'm1_mappo_actor.pt'))
torch.save(m1_critic.state_dict(), os.path.join(SAVE_DIR, 'm1_mappo_critic.pt'))
for m in MODE_NAMES:
    torch.save(m2_actors[m].state_dict(), os.path.join(SAVE_DIR, f'm2_pareto_actor_{m}.pt'))
torch.save(m2_critic.state_dict(),    os.path.join(SAVE_DIR, 'm2_pareto_critic.pt'))
torch.save(m3_actor.state_dict(),     os.path.join(SAVE_DIR, 'm3_m2llm_actor.pt'))
torch.save(m3_critic.state_dict(),    os.path.join(SAVE_DIR, 'm3_m2llm_critic.pt'))
print('All weights saved to', os.path.abspath(SAVE_DIR))